# AgriNex Disease ML: Notebook 01 - Exploratory Data Analysis & Data Preparation

**Objective**: Audit, inspect, verify integrity, and prepare plant/crop leaf disease datasets for model training.

### Tasks Covered:
1. Installing & importing dependencies
2. Loading dataset (Google Drive / Kaggle / Local)
3. Inspecting target classes & crop condition labels
4. Counting images per class (class distribution analysis)
5. Displaying sample leaf images across classes
6. Auditing image sizes, resolutions, color spaces, and aspect ratios
7. Detecting zero-byte & corrupted images
8. Preparing stratified train / validation / test splits & metadata CSVs

--- 
## 1. Installing & Importing Dependencies

In [ ]:
# Install required packages if running in Google Colab environment
import sys
if 'google.colab' in sys.modules:
    !pip install -q torch torchvision opencv-python pillow pandas numpy matplotlib seaborn albumentations tqdm gdown scikit-learn

import os
import random
from pathlib import Path
from typing import List, Dict, Tuple
from PIL import Image, ImageFile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# Enable loading truncated images safely during inspection
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Set display parameters
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("✅ All dependencies successfully imported!")

--- 
## 2. Loading the Dataset

Mount Google Drive or download the PlantVillage / AgriNex disease dataset into `data/raw`.

In [ ]:
# Mount Google Drive if running in Google Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    # Uncomment below to mount drive:
    # drive.mount('/content/drive')
    pass

# Define project paths relative to notebook location
BASE_DIR = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
RAW_DATA_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw Data Directory: {RAW_DATA_DIR}")
print(f"Processed Data Directory: {PROCESSED_DATA_DIR}")

# Helper: Generate demo sample structure if data directory is empty
if not any(RAW_DATA_DIR.iterdir()) or len(list(RAW_DATA_DIR.glob('*/*'))) == 0:
    print("ℹ️ RAW_DATA_DIR is currently empty. Creating starter dummy structure for testing execution...")
    dummy_classes = ["Tomato___Early_blight", "Tomato___Late_blight", "Tomato___healthy", "Potato___Early_blight", "Potato___healthy"]
    for cls in dummy_classes:
        cls_dir = RAW_DATA_DIR / cls
        cls_dir.mkdir(parents=True, exist_ok=True)
        for i in range(15):
            dummy_img = Image.fromarray(np.uint8(np.random.randint(0, 255, (224, 224, 3))))
            dummy_img.save(cls_dir / f"sample_{i:03d}.jpg")
    print("✅ Created sample images for verification.")

--- 
## 3. Inspecting Classes

In [ ]:
class_folders = sorted([d.name for d in RAW_DATA_DIR.iterdir() if d.is_dir() and not d.name.startswith('.')])
print(f"📌 Total Classes Found: {len(class_folders)}\n")

healthy_classes = [c for c in class_folders if 'healthy' in c.lower()]
diseased_classes = [c for c in class_folders if 'healthy' not in c.lower()]

print(f"🟢 Healthy Condition Classes ({len(healthy_classes)}):")
for c in healthy_classes:
    print(f"   - {c}")

print(f"\n🔴 Diseased Condition Classes ({len(diseased_classes)}):")
for c in diseased_classes:
    print(f"   - {c}")

--- 
## 4. Counting Images per Class

In [ ]:
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
class_counts = []

for cls in class_folders:
    cls_dir = RAW_DATA_DIR / cls
    images = [f for f in cls_dir.glob('*') if f.suffix.lower() in VALID_EXTENSIONS]
    class_counts.append({
        'class_name': cls,
        'image_count': len(images),
        'is_healthy': 'healthy' in cls.lower()
    })

df_counts = pd.DataFrame(class_counts).sort_values(by='image_count', ascending=False)
df_counts['percentage'] = (df_counts['image_count'] / df_counts['image_count'].sum()) * 100

print("📊 Class Distribution Summary:")
display(df_counts)

# Plot Class Distribution
plt.figure(figsize=(12, max(6, len(class_folders) * 0.4)))
palette = ['#2ecc71' if is_h else '#e74c3c' for is_h in df_counts['is_healthy']]
ax = sns.barplot(data=df_counts, y='class_name', x='image_count', palette=palette)
plt.title('Crop Disease Dataset - Image Counts per Class', fontsize=14, fontweight='bold')
plt.xlabel('Number of Images')
plt.ylabel('Class Label')
for p in ax.patches:
    width = p.get_width()
    ax.annotate(f'{int(width)}', (width + 5, p.get_y() + p.get_height() / 2.), va='center')
plt.tight_layout()
plt.show()

--- 
## 5. Displaying Sample Images per Class

In [ ]:
samples_per_class = min(3, len(class_folders))
num_classes_to_show = min(6, len(class_folders))
selected_classes = random.sample(class_folders, num_classes_to_show)

fig, axes = plt.subplots(num_classes_to_show, samples_per_class, figsize=(12, 3 * num_classes_to_show))
if num_classes_to_show == 1:
    axes = np.expand_dims(axes, axis=0)

for r_idx, cls_name in enumerate(selected_classes):
    cls_dir = RAW_DATA_DIR / cls_name
    images = [f for f in cls_dir.glob('*') if f.suffix.lower() in VALID_EXTENSIONS]
    sample_imgs = random.sample(images, min(samples_per_class, len(images)))
    
    for c_idx in range(samples_per_class):
        ax = axes[r_idx, c_idx]
        if c_idx < len(sample_imgs):
            img_path = sample_imgs[c_idx]
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(f"{cls_name[:20]}\n{img.size[0]}x{img.size[1]}", fontsize=9)
        ax.axis('off')

plt.suptitle('Sample Leaf Images Across Crop Disease Categories', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

--- 
## 6. Checking Image Sizes & Resolutions

In [ ]:
image_metadata = []

for cls in tqdm(class_folders, desc="Auditing Image Sizes"):
    cls_dir = RAW_DATA_DIR / cls
    images = [f for f in cls_dir.glob('*') if f.suffix.lower() in VALID_EXTENSIONS]
    
    for img_p in images:
        try:
            with Image.open(img_p) as img:
                w, h = img.size
                mode = img.mode
                image_metadata.append({
                    'path': str(img_p),
                    'class': cls,
                    'width': w,
                    'height': h,
                    'mode': mode,
                    'channels': len(img.getbands())
                })
        except Exception as e:
            pass

df_meta = pd.DataFrame(image_metadata)
print(f"Total Images Audited: {len(df_meta)}\n")
print("Resolution Summary Stats:")
display(df_meta[['width', 'height', 'channels']].describe())

print("\nColor Modes Found:", df_meta['mode'].value_counts().to_dict())

# Resolution Distribution Histograms
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_meta['width'], kde=True, ax=ax1, color='teal')
ax1.set_title('Image Width Distribution')
sns.histplot(df_meta['height'], kde=True, ax=ax2, color='coral')
ax2.set_title('Image Height Distribution')
plt.tight_layout()
plt.show()

--- 
## 7. Detecting Obvious Corrupted Images

In [ ]:
corrupt_files = []
zero_byte_files = []

for cls in tqdm(class_folders, desc="Scanning for Corrupted Files"):
    cls_dir = RAW_DATA_DIR / cls
    files = list(cls_dir.glob('*'))
    
    for f in files:
        if f.suffix.lower() not in VALID_EXTENSIONS:
            continue
        
        # Check zero-byte file
        if f.stat().st_size == 0:
            zero_byte_files.append(str(f))
            continue
            
        # PIL verify
        try:
            with Image.open(f) as img:
                img.verify()
        except Exception as err:
            corrupt_files.append((str(f), str(err)))

print(f"\n🔍 Corruption Scan Results:")
print(f"   - Zero-Byte Files Found: {len(zero_byte_files)}")
print(f"   - Unreadable/Corrupt Image Files Found: {len(corrupt_files)}")

if corrupt_files:
    print("⚠️ Corrupted Files List:")
    for cf, err in corrupt_files[:5]:
        print(f"   {cf} -> Error: {err}")
else:
    print("✅ No corrupted image files detected in dataset.")

--- 
## 8. Preparing the Dataset for Training

In [ ]:
# Create Stratified Train / Validation / Test Metadata Split
if not df_meta.empty:
    train_df, temp_df = train_test_split(df_meta, test_size=0.3, random_state=42, stratify=df_meta['class'])
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['class'])
    
    print(f"📦 Dataset Split Summary:")
    print(f"   - Train set: {len(train_df)} samples ({len(train_df)/len(df_meta)*100:.1f}%)")
    print(f"   - Validation set: {len(val_df)} samples ({len(val_df)/len(df_meta)*100:.1f}%)")
    print(f"   - Test set: {len(test_df)} samples ({len(test_df)/len(df_meta)*100:.1f}%)")
    
    # Save metadata CSV files
    train_df.to_csv(PROCESSED_DATA_DIR / "train_split.csv", index=False)
    val_df.to_csv(PROCESSED_DATA_DIR / "val_split.csv", index=False)
    test_df.to_csv(PROCESSED_DATA_DIR / "test_split.csv", index=False)
    print(f"\n✅ Saved split metadata to {PROCESSED_DATA_DIR}")

--- 
## 📌 Notebook Summary

- Dependencies installed.
- Dataset classes inspected.
- Image counts per class calculated and visualized.
- Image sizes and resolution statistics checked.
- Corrupted images scanned.
- Train / Val / Test metadata split prepared.

**Status**: Ready for model architecture development & PyTorch model training!